# ⚾ Pitcher Injury Risk Analysis — BC PBL Hackathon 2026

**Goal:** Identify pitchers at highest injury risk heading into the playoffs, based solely on regular-season workload profiles.

**Levels:** M (Major/highest), A (Advanced/medium), R (Rookie/lowest)  
**Dataset:** 851 pitch-by-pitch game CSVs + `player_pitching.csv` + `injuries.csv`

---
### Pipeline (modular functions in `pitcher_injury_risk.py`)
| Step | Function | What it does |
|------|----------|--------------|
| 1 | `load_all_files()` | Recursively reads all CSV/Excel files |
| 2 | `build_pitcher_game_table()` | Aggregates pitch counts per pitcher per game |
| 3 | `engineer_features()` | Computes 20+ workload features per pitcher |
| 4 | `compute_risk_scores()` | Level-stratified 0–100 score + Low/Moderate/High label |
| 5 | `summarize_and_export()` | Merges metadata, writes `pitcher_risk_summary.csv` |
| — | `explain_pitcher_risk(id)` | Coach-friendly narrative for any pitcher |

---
### Risk Score Components
| Component | Weight | Driver |
|-----------|--------|--------|
| Seasonal pitch volume | 20% | High volume → cumulative fatigue |
| High-pitch outings (>80p / >95p) | 20% | Acute single-game stress |
| Peak ACWR spike (>1.5) | 20% | Workload spikes correlate with injury |
| Short-rest frequency (<3 / <4 days) | 15% | Insufficient recovery time |
| End-season workload (last 7 / 14 days) | 15% | Fatigue entering playoffs |
| Peak single-outing load | 10% | Max pitch count in one game |

> All components are **min-max normalised within level** to ensure M / A / R pitchers are compared only against their peers.

## 0. Setup

In [17]:
import sys
from pathlib import Path

# Add the folder to sys.path so we can import the analysis module
FOLDER_PATH = Path("/Users/pranayratan/Desktop/AdvancedLevel_SSAC Sport Med Hackathon 2026 copy")
sys.path.insert(0, str(FOLDER_PATH))

import warnings, pandas as pd, numpy as np, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

# Import all pipeline functions from the companion script
from pitcher_injury_risk import (
    load_all_files, build_pitcher_game_table, engineer_features,
    compute_risk_scores, summarize_and_export, explain_pitcher_risk,
    plot_risk_distributions, plot_workload_timeseries,
    FOLDER_PATH, OUTPUT_PATH, GAME_PATTERN,
    THRESH_80, THRESH_95, ACWR_SPIKE, LEVEL_LABEL
)
print(f"Folder : {FOLDER_PATH}")
print(f"Outputs: {OUTPUT_PATH}")

Folder : /Users/pranayratan/Desktop/AdvancedLevel_SSAC Sport Med Hackathon 2026 copy
Outputs: /Users/pranayratan/Desktop/AdvancedLevel_SSAC Sport Med Hackathon 2026 copy/outputs


## 1. Load Data

In [18]:
raw_files  = load_all_files(FOLDER_PATH)
game_files = {k: v for k, v in raw_files.items() if GAME_PATTERN.search(k)}
player_df  = raw_files.get(str(FOLDER_PATH / 'player_pitching.csv'), pd.DataFrame())
injury_df  = raw_files.get(str(FOLDER_PATH / 'injuries.csv'),        pd.DataFrame())

print(f"Game CSVs : {len(game_files)}")
print(f"Players   : {len(player_df)}")
print(f"Injuries  : {len(injury_df)}")
print()
display(player_df.head())

Loaded 855 files.
Game CSVs : 851
Players   : 364
Injuries  : 6



,PitcherId,Name,Team,Role,G,GS,IP,BF,Pitches,H,2B,3B,HR,BB,K,R,ER,K9,BB9,WHIP,ERA,KBB%,FIP
0,P01008,Charlie Moss,1,RP,25,0,31.10,133,498,18,1,0,4,20,48,7,7,13.79,5.74,1.21,2.01,21.10,2.72
1,P01013,Chris Sawyer,1,RP,7,0,7.00,26,101,3,0,0,0,2,13,0,0,16.71,2.57,0.71,0.00,42.30,-0.65
2,P01007,Elliot Holland,1,RP,58,0,63.20,291,1165,49,6,3,8,52,95,25,25,13.43,7.35,1.59,3.53,14.80,3.30
3,P01012,Hayden Griffin,1,RP,3,0,3.00,17,77,2,0,0,0,6,3,1,1,9.00,18.00,2.67,3.00,-17.60,6.21
4,P01002,Hayden Reed,1,SP,54,54,47.00,255,1018,49,14,0,10,56,100,29,29,19.15,10.72,2.23,5.55,17.30,4.29


## 2. Build Pitcher–Game Table

In [19]:
pitcher_games = build_pitcher_game_table(game_files)
print(f"Rows      : {len(pitcher_games):,}")
print(f"Pitchers  : {pitcher_games['pitcher_id'].nunique():,}")
print(f"Date range: {pitcher_games['game_date'].min().date()} → {pitcher_games['game_date'].max().date()}")
print(f"Level distribution:")
print(pitcher_games['level'].value_counts().to_string())
print()
display(pitcher_games.head(10))

Rows      : 3,440
Pitchers  : 364
Date range: 2026-04-05 → 2026-07-24
Level distribution:
level
A    1184
R    1135
M    1121



,pitcher_id,pitcher_name,team,pitches,game_id,game_date,home_team,away_team,level,prev_date,days_rest
0,P01001,Rowan Griffin,37,120,M0008,2026-04-07,37,1,M,NaT,NaN
1,P01001,Rowan Griffin,22,124,M0041,2026-04-16,1,22,M,2026-04-07,9.00
2,P01001,Rowan Griffin,7,120,M0069,2026-04-28,7,1,M,2026-04-16,12.00
3,P01001,Rowan Griffin,28,144,M0105,2026-05-12,28,1,M,2026-04-28,14.00
4,P01001,Rowan Griffin,13,122,M0137,2026-05-21,1,13,M,2026-05-12,9.00
5,P01001,Rowan Griffin,22,123,M0165,2026-06-03,1,22,M,2026-05-21,13.00
6,P01001,Rowan Griffin,25,131,M0195,2026-06-16,25,1,M,2026-06-03,13.00
7,P01001,Rowan Griffin,19,142,M0231,2026-06-25,1,19,M,2026-06-16,9.00
8,P01001,Rowan Griffin,25,142,M0262,2026-07-09,1,25,M,2026-06-25,14.00
9,P01002,Hayden Reed,34,112,M0017,2026-04-08,1,34,M,NaT,NaN


## 3. Feature Engineering

In [20]:
features, pitcher_games_enriched = engineer_features(pitcher_games)
print(f"Feature table: {features.shape} (one row per pitcher)")
print()
display(features.head())

Feature table: (364, 24) (one row per pitcher)



,pitcher_id,pitcher_name,team,level,Role,outings,total_pitches,max_pitches_outing,mean_pitches_outing,std_pitches_outing,gt75_outings,gt80_outings,gt95_outings,short_rest_lt3,short_rest_lt4,mean_rest_days,median_rest_days,max_acwr,acwr_spikes,end_pitches_7d,end_pitches_14d,end_outings_7d,end_outings_14d,back_to_back_late
0,P01001,Rowan Griffin,37,M,SP,9,1168,144,129.78,10.21,9,9,9,0,0,11.62,12.50,1.48,0,0,0,0,0,0
1,P01002,Hayden Reed,34,M,SP,9,1018,138,113.11,30.75,8,8,7,0,0,11.75,13.00,2.03,1,0,132,0,1,0
2,P01003,Logan Young,31,M,SP,9,1150,140,127.78,11.16,9,9,9,0,0,12.12,13.00,1.48,0,0,140,0,1,0
3,P01004,Micah Holland,28,M,SP,9,1130,142,125.56,12.33,9,9,9,0,0,11.62,13.00,1.50,0,0,141,0,1,0
4,P01005,Kai Porter,25,M,SP,8,1013,141,126.62,8.83,8,8,8,0,0,12.00,13.00,1.45,0,0,0,0,0,0


In [21]:
# Descriptive statistics for key risk features
key_cols = [
    'total_pitches', 'outings', 'max_pitches_outing',
    'gt80_outings', 'gt95_outings',
    'short_rest_lt3', 'short_rest_lt4',
    'max_acwr', 'acwr_spikes',
    'end_pitches_7d', 'end_outings_7d'
]
display(features[key_cols].describe().round(2))

,total_pitches,outings,max_pitches_outing,gt80_outings,gt95_outings,short_rest_lt3,short_rest_lt4,max_acwr,acwr_spikes,end_pitches_7d,end_outings_7d
count,364.00,364.00,364.00,364.00,364.00,364.00,364.00,282.00,364.00,364.00,364.00
mean,754.21,9.26,107.47,5.45,3.95,0.91,1.05,2.35,2.16,21.18,0.26
std,499.89,6.01,39.01,4.69,3.82,1.79,2.02,0.81,2.25,44.70,0.53
min,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.28,0.00,0.00,0.00
25%,141.75,2.00,87.50,1.00,0.00,0.00,0.00,1.77,0.00,0.00,0.00
50%,908.00,9.00,108.50,4.50,3.00,0.00,0.00,2.15,1.00,0.00,0.00
75%,1140.50,14.00,133.00,9.00,7.00,1.00,1.00,2.68,4.00,0.00,0.00
max,1661.00,24.00,207.00,16.00,13.00,11.00,12.00,4.00,9.00,266.00,3.00


## 4. Risk Scoring (level-stratified, 0–100)

Each component is **min-max normalised within level** before weighting, so M-level starters are not unfairly penalised for throwing more total pitches than Rookie-level relievers.

In [22]:
risk_df = compute_risk_scores(features)

print('=== Risk label distribution ===')
dist = risk_df.groupby(['level', 'risk_label']).size().unstack(fill_value=0)
dist = dist.reindex(columns=['Low', 'Moderate', 'High'], fill_value=0)
display(dist)

at_risk = risk_df[risk_df['risk_label'].ne('Low')]
print(f"\nTotal at-risk (Moderate + High): {len(at_risk)} pitchers")

=== Risk label distribution ===


risk_label,Low,Moderate,High
level,,,
A,63,82,4
M,63,76,3
R,35,33,5



Total at-risk (Moderate + High): 203 pitchers


## 5. Export Summary CSV

In [26]:
summary = summarize_and_export(risk_df, player_df, OUTPUT_PATH)
print(f"Summary table: {summary.shape}")
display(summary.sort_values('risk_score_0_100', ascending=False).head(20))

✅  Saved: /Users/pranayratan/Desktop/AdvancedLevel_SSAC Sport Med Hackathon 2026 copy/outputs/pitcher_risk_summary.csv
Summary table: (364, 30)


,pitcher_name,team,org,level,Role,outings,GS,total_pitches,IP,max_pitches_outing,mean_pitches_outing,std_pitches_outing,gt75_outings,gt80_outings,gt95_outings,short_rest_lt3,short_rest_lt4,mean_rest_days,median_rest_days,max_acwr,acwr_spikes,end_pitches_7d,end_pitches_14d,end_outings_7d,end_outings_14d,back_to_back_late,ERA,FIP,risk_score_0_100,risk_label
pitcher_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
P18006,Micah Reed,18,18,R,RP,15,77,1660,102.10,150,83.00,36.46,12,11,9,5,7,5.16,7.00,2.71,5,141,314,1,3,1,1.50,2.14,85.40,High
P23006,Taylor Lawson,23,23,A,RP,20,0,1155,66.00,191,57.75,44.71,7,5,3,6,6,5.21,6.00,2.24,5,0,191,0,1,0,0.95,2.11,83.40,High
P07006,Casey Bennett,7,7,M,RP,21,0,1257,47.20,164,59.86,38.52,7,7,4,6,8,5.00,6.00,2.49,5,0,164,0,1,0,11.90,7.24,83.00,High
P05006,Reese Quinn,5,5,A,RP,21,0,1053,63.20,145,50.14,39.04,4,4,4,6,6,4.90,5.00,3.43,6,8,147,1,4,1,1.13,1.73,82.40,High
P37002,Jesse Cooper,1,37,M,SP,13,107,1618,100.00,145,124.46,26.97,12,12,12,0,0,8.17,7.00,1.93,1,0,145,0,1,0,1.89,1.13,76.30,High
P05001,Logan Ramsey,38,5,A,SP,14,105,1661,95.20,135,118.64,14.35,14,14,13,0,0,8.08,7.00,1.61,1,113,243,1,2,0,1.79,1.65,75.50,High
P30001,Sawyer Lawson,15,30,R,SP,16,109,1559,95.20,164,97.44,20.78,16,16,8,0,0,7.00,7.00,2.35,2,266,266,2,2,0,1.60,1.08,74.90,High
P03006,Elliot Hunter,3,3,R,RP,15,55,1492,94.20,179,78.53,42.49,10,10,7,5,5,5.39,6.50,2.70,9,0,218,0,3,1,0.95,1.11,72.30,High
P01007,Elliot Holland,1,1,M,RP,21,0,1165,63.20,143,55.48,43.02,7,7,6,6,6,4.95,5.00,1.90,2,0,93,0,1,0,3.53,3.30,72.10,High


## 6. At-Risk Pitchers by Level

In [27]:
display_cols = [
    'pitcher_name', 'team', 'level', 'Role',
    'outings', 'total_pitches', 'max_pitches_outing',
    'short_rest_lt3', 'max_acwr', 'acwr_spikes',
    'end_pitches_7d', 'risk_score_0_100', 'risk_label'
]

for lvl in ['M', 'A', 'R']:
    sub = summary[
        (summary['level'] == lvl) &
        (summary['risk_label'].isin(['Moderate', 'High']))
    ].sort_values('risk_score_0_100', ascending=False)

    print(f"\n{'='*70}")
    print(f"  {LEVEL_LABEL[lvl]} — {len(sub)} pitchers at Moderate/High risk")
    print(f"{'='*70}")
    cols = [c for c in display_cols if c in sub.columns]
    display(sub[cols].head(20))


  Major (M) — 79 pitchers at Moderate/High risk


,pitcher_name,team,level,Role,outings,total_pitches,max_pitches_outing,short_rest_lt3,max_acwr,acwr_spikes,end_pitches_7d,risk_score_0_100,risk_label
pitcher_id,,,,,,,,,,,,,
P07006,Casey Bennett,7,M,RP,21,1257,164,6,2.49,5,0,83.00,High
P37002,Jesse Cooper,1,M,SP,13,1618,145,0,1.93,1,0,76.30,High
P01007,Elliot Holland,1,M,RP,21,1165,143,6,1.90,2,0,72.10,High
P19006,Logan Holland,19,M,RP,21,1056,135,6,3.77,6,0,69.30,Moderate
P28001,Elliot Carter,13,M,SP,11,1421,146,0,1.90,1,0,69.20,Moderate
P34007,Evan Reed,34,M,RP,14,917,136,2,3.12,6,0,67.90,Moderate
P34002,Nolan Clayton,1,M,SP,11,1457,145,0,1.47,0,0,67.90,Moderate
P16003,Hayden Manning,13,M,SP,9,1223,158,0,1.93,1,0,66.80,Moderate
P25001,Hayden Quinn,16,M,SP,11,1396,143,0,1.53,2,0,66.40,Moderate



  Advanced (A) — 86 pitchers at Moderate/High risk


,pitcher_name,team,level,Role,outings,total_pitches,max_pitches_outing,short_rest_lt3,max_acwr,acwr_spikes,end_pitches_7d,risk_score_0_100,risk_label
pitcher_id,,,,,,,,,,,,,
P23006,Taylor Lawson,23,A,RP,20,1155,191,6,2.24,5,0,83.40,High
P05006,Reese Quinn,5,A,RP,21,1053,145,6,3.43,6,8,82.40,High
P05001,Logan Ramsey,38,A,SP,14,1661,135,0,1.61,1,113,75.50,High
P38006,Quinn Porter,38,A,RP,19,983,102,5,3.55,7,34,72.00,High
P02006,Evan Parker,2,A,RP,19,1061,138,4,2.14,5,67,67.60,Moderate
P23007,Cole Carson,23,A,RP,18,771,101,4,3.36,7,48,67.10,Moderate
P32006,Elliot Kennedy,32,A,RP,17,1008,103,1,2.56,6,132,66.60,Moderate
P17006,Charlie Manning,17,A,RP,21,927,82,6,2.58,4,82,65.50,Moderate
P26004,Finley Carter,5,A,SP,11,1385,137,0,1.46,0,134,65.20,Moderate



  Rookie (R) — 38 pitchers at Moderate/High risk


,pitcher_name,team,level,Role,outings,total_pitches,max_pitches_outing,short_rest_lt3,max_acwr,acwr_spikes,end_pitches_7d,risk_score_0_100,risk_label
pitcher_id,,,,,,,,,,,,,
P18006,Micah Reed,18,R,RP,15,1660,150,5,2.71,5,141,85.40,High
P30001,Sawyer Lawson,15,R,SP,16,1559,164,0,2.35,2,266,74.90,High
P03006,Elliot Hunter,3,R,RP,15,1492,179,5,2.70,9,0,72.30,High
P27003,Hayden Spencer,12,R,RP,14,1283,140,4,2.74,6,174,71.50,High
P27004,Luca Spencer,6,R,RP,16,1547,120,6,2.12,5,182,70.70,High
P09006,Archer Parker,9,R,RP,13,1165,196,3,2.97,6,95,64.10,Moderate
P06003,Mason Quinn,27,R,RP,18,1375,185,2,2.12,3,200,64.00,Moderate
P27006,Jesse Holland,27,R,RP,16,1361,122,4,2.50,5,86,63.50,Moderate
P24006,Micah Bennett,24,R,RP,12,1306,152,4,2.67,7,0,60.40,Moderate


## 7. `explain_pitcher_risk()` — Coach-Friendly Narratives

Call `explain_pitcher_risk('<pitcher_id>', summary)` to print a complete narrative
for any pitcher. The cell below demos the top 5 highest-risk pitchers.

In [29]:
top5_ids = (
    summary[summary['risk_label'].isin(['Moderate', 'High'])]
    .sort_values('risk_score_0_100', ascending=False)
    .head(5)
    .index
)

for pid in top5_ids:
    explain_pitcher_risk(pid, summary)
    print()

🔴  Micah Reed  |  Team 18  |  Level R  |  High Risk  |  Score 85.4/100
═════════════════════════════════════════════════════════════════
  Role / IP         : RP / 102.1 IP
  Season volume     : 1660 pitches across 15 outings
  Peak outing       : 150 pitches  (avg 83.0 ± 36.5)
  High-pitch games  : 11 above 80 pitches, 9 above 95 pitches
                      → repeated high-effort outings raise cumulative tissue stress.
  Short rest        : 5× on <3 days rest, 7× on <4 days rest  (mean 5.2d)
                      → insufficient recovery amplifies injury risk.
  Peak ACWR         : 2.71  ⚠️  DANGER ZONE
                      (5 outing(s) above the 1.5 spike threshold)
  End-season push   : 141 pitches / 1 outings in last 7 days
                      314 pitches / 3 outings in last 14 days
                      → reduced recovery buffer before playoffs.
  Back-to-back late : 1 outing(s) on ≤1 day rest in last 2 weeks
                      → significant cumulative fatigue risk.
═══════

In [32]:
# Interactive: paste any pitcher_id here to get their explanation
explain_pitcher_risk('P18003', summary)   # ← change this ID

🟡  Luca Lawson  |  Team 21  |  Level R  |  Moderate Risk  |  Score 53.4/100
═════════════════════════════════════════════════════════════════
  Role / IP         : RP / 74.1 IP
  Season volume     : 1414 pitches across 17 outings
  Peak outing       : 138 pitches  (avg 74.4 ± 24.6)
  High-pitch games  : 9 above 80 pitches, 3 above 95 pitches
                      → repeated high-effort outings raise cumulative tissue stress.
  Short rest        : 3× on <3 days rest, 3× on <4 days rest  (mean 5.4d)
                      → insufficient recovery amplifies injury risk.
  Peak ACWR         : 2.27  ⚠️  DANGER ZONE
                      (9 outing(s) above the 1.5 spike threshold)
  End-season push   : no outings in final 7 days ✓
  Back-to-back late : 1 outing(s) on ≤1 day rest in last 2 weeks
                      → significant cumulative fatigue risk.
═════════════════════════════════════════════════════════════════


'🟡  Luca Lawson  |  Team 21  |  Level R  |  Moderate Risk  |  Score 53.4/100\n═════════════════════════════════════════════════════════════════\n  Role / IP         : RP / 74.1 IP\n  Season volume     : 1414 pitches across 17 outings\n  Peak outing       : 138 pitches  (avg 74.4 ± 24.6)\n  High-pitch games  : 9 above 80 pitches, 3 above 95 pitches\n                      → repeated high-effort outings raise cumulative tissue stress.\n  Short rest        : 3× on <3 days rest, 3× on <4 days rest  (mean 5.4d)\n                      → insufficient recovery amplifies injury risk.\n  Peak ACWR         : 2.27  ⚠️  DANGER ZONE\n                      (9 outing(s) above the 1.5 spike threshold)\n  End-season push   : no outings in final 7 days ✓\n  Back-to-back late : 1 outing(s) on ≤1 day rest in last 2 weeks\n                      → significant cumulative fatigue risk.\n═════════════════════════════════════════════════════════════════'

## 8. Visualizations

In [33]:
import matplotlib
matplotlib.use('inline' if 'inline' in str(matplotlib.get_backend()) else 'Agg')

PALETTE = {'M': '#E74C3C', 'A': '#F39C12', 'R': '#2ECC71'}

# ── Plot 1: Risk score histograms by level ─────────────────────────────────────
summary_reset = summary.reset_index()
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Pitcher Injury-Risk Score Distribution by Level',
             fontsize=15, fontweight='bold', y=1.02)

for ax, lvl in zip(axes, ['M', 'A', 'R']):
    sub = summary_reset[summary_reset['level'] == lvl]['risk_score_0_100']
    ax.hist(sub, bins=20, color=PALETTE[lvl], edgecolor='white', alpha=0.85, lw=0.6)
    ax.axvline(40, color='orange', lw=1.5, ls='--', label='Moderate (40)')
    ax.axvline(70, color='red',    lw=1.5, ls='--', label='High (70)')
    ax.set_title(LEVEL_LABEL[lvl], fontsize=12, fontweight='bold')
    ax.set_xlabel('Risk Score (0–100)')
    ax.set_ylabel('# Pitchers')
    ax.legend(fontsize=8)
    n_high = (sub >= 70).sum()
    n_mod  = ((sub >= 40) & (sub < 70)).sum()
    ax.text(0.97, 0.93, f'High={n_high}  Mod={n_mod}',
            transform=ax.transAxes, ha='right', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'risk_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [34]:
# ── Plot 2: Stacked bar — risk label counts per level ──────────────────────────
label_counts = (
    summary_reset.groupby(['level', 'risk_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['Low', 'Moderate', 'High'], fill_value=0)
    .reindex(['M', 'A', 'R'])
)

fig2, ax2 = plt.subplots(figsize=(8, 5))
label_counts.plot(
    kind='bar', stacked=True, ax=ax2,
    color=['#2ECC71', '#F39C12', '#E74C3C'],
    edgecolor='white', linewidth=0.5
)
ax2.set_xticklabels([LEVEL_LABEL[l] for l in label_counts.index], rotation=0)
ax2.set_ylabel('Number of Pitchers')
ax2.set_title('Risk Label Counts per Level', fontsize=13, fontweight='bold')
ax2.legend(title='Risk Label', loc='upper right')
for c in ax2.containers:
    ax2.bar_label(c, label_type='center', fontsize=9, color='white', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'risk_label_counts.png', dpi=150, bbox_inches='tight')
plt.show()

In [35]:
# ── Plot 3: Workload time-series — High-risk vs Low-risk pitcher ───────────────
import matplotlib.dates as mdates

game_pids  = set(pitcher_games_enriched['pitcher_id'].unique())
mod_cands  = summary[summary['risk_label'] == 'Moderate'].sort_values('risk_score_0_100', ascending=False)
low_cands  = summary[summary['risk_label'] == 'Low'].sort_values('risk_score_0_100')
high_pid   = mod_cands[mod_cands.index.isin(game_pids)].index[0]
low_pid    = low_cands[low_cands.index.isin(game_pids)].index[0]

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Workload Profiles: Highest-Risk vs Lowest-Risk Pitcher', fontsize=14, fontweight='bold')

for col, pid, risk_tag in [(0, high_pid, 'Highest Risk'), (1, low_pid, 'Lowest Risk')]:
    g = pitcher_games_enriched[pitcher_games_enriched['pitcher_id'] == pid].sort_values('game_date')
    name  = summary.loc[pid, 'pitcher_name'] if pid in summary.index else pid
    score = summary.loc[pid, 'risk_score_0_100'] if pid in summary.index else '?'
    color = '#E74C3C' if col == 0 else '#2ECC71'

    ax0 = axes[0, col]
    ax0.bar(g['game_date'], g['pitches'], color=color, alpha=0.75, width=1.5)
    ax0.axhline(THRESH_80, color='orange', ls='--', lw=1.2, label=f'{THRESH_80}p threshold')
    ax0.axhline(THRESH_95, color='red',    ls='--', lw=1.2, label=f'{THRESH_95}p threshold')
    ax0.set_title(f'{name} — {risk_tag} (score={score})', fontweight='bold')
    ax0.set_ylabel('Pitches per Outing')
    ax0.legend(fontsize=8)
    ax0.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax0.get_xticklabels(), rotation=30, ha='right')

    ax1 = axes[1, col]
    acwr = g[['game_date', 'acwr']].dropna()
    if not acwr.empty:
        ax1.plot(acwr['game_date'], acwr['acwr'], color=color, lw=2, marker='o', ms=4)
        ax1.axhline(ACWR_SPIKE, color='red', ls='--', lw=1.2, label=f'ACWR={ACWR_SPIKE}')
        ax1.fill_between(acwr['game_date'], acwr['acwr'], ACWR_SPIKE,
                          where=acwr['acwr'] > ACWR_SPIKE, color='red', alpha=0.2, label='Spike')
    ax1.set_ylabel('ACWR (7d / 28d chronic)')
    ax1.set_xlabel('Date')
    ax1.legend(fontsize=8)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.setp(ax1.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(OUTPUT_PATH / 'workload_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Known Injuries Cross-Check

In [36]:
if not injury_df.empty:
    inj   = injury_df.rename(columns={'PitcherId': 'pitcher_id'})
    cross = inj.merge(
        summary[['pitcher_name', 'risk_score_0_100', 'risk_label']].reset_index(),
        on='pitcher_id', how='left'
    )
    display(cross[['pitcher_id', 'Name', 'Date', 'GameID', 'PitchesThisGame',
                   'Limit', 'OverBy', 'InjuryDays', 'risk_score_0_100', 'risk_label']])

    flagged = cross[cross['risk_label'].isin(['Moderate', 'High'])]
    print(f"\n{len(flagged)} / {len(cross)} known-injured pitchers flagged Moderate/High.")
    print()
    print("📝 ANALYTICAL NOTE:")
    print("   All 6 recorded injuries occurred DURING the regular season (Apr–Jul).")
    print("   Those pitchers subsequently missed games, lowering their end-of-season")
    print("   totals (total pitches, ACWR, etc.) relative to healthy pitchers.")
    print("   This is expected and not a model failure — our goal is END-OF-SEASON")
    print("   pre-playoff risk flagging, not in-season event prediction.")
    print("   The pitchers we flag Moderate/High are those who have the largest")
    print("   CUMULATIVE FATIGUE BURDEN entering the playoffs.")
else:
    print('injuries.csv not loaded.')

,pitcher_id,Name,Date,GameID,PitchesThisGame,Limit,OverBy,InjuryDays,risk_score_0_100,risk_label
0,P17002,Evan Jensen,2026-04-09,A0010,96,67,29,25,1.70,Low
1,P05004,Nolan Foster,2026-04-10,A0018,91,59,32,18,0.00,Low
2,P37001,Cole Griffin,2026-04-14,M0026,123,95,28,23,1.70,Low
3,P25002,Elliot Douglas,2026-04-29,M0075,116,71,45,31,15.90,Low
4,P02003,Drew Moss,2026-05-06,A0088,113,67,46,24,20.70,Low
5,P21003,Jordan Quinn,2026-07-09,R0244,58,27,31,24,42.50,Moderate



1 / 6 known-injured pitchers flagged Moderate/High.

📝 ANALYTICAL NOTE:
   All 6 recorded injuries occurred DURING the regular season (Apr–Jul).
   Those pitchers subsequently missed games, lowering their end-of-season
   totals (total pitches, ACWR, etc.) relative to healthy pitchers.
   This is expected and not a model failure — our goal is END-OF-SEASON
   pre-playoff risk flagging, not in-season event prediction.
   The pitchers we flag Moderate/High are those who have the largest
   CUMULATIVE FATIGUE BURDEN entering the playoffs.


## 10. Final Summary

In [37]:
print('\n' + '='*65)
print('  FINAL PITCHER RISK SUMMARY'.center(65))
print('='*65)

for lvl in ['M', 'A', 'R']:
    sub  = summary[summary['level'] == lvl]
    high = (sub['risk_label'] == 'High').sum()
    mod  = (sub['risk_label'] == 'Moderate').sum()
    low  = (sub['risk_label'] == 'Low').sum()
    print(f"  {LEVEL_LABEL[lvl]:20s} → 🔴 High: {high:3d}  🟡 Moderate: {mod:3d}  🟢 Low: {low:3d}")

print(f"\n  Output CSV : {OUTPUT_PATH / 'pitcher_risk_summary.csv'}")
print(f"  Plot 1     : {OUTPUT_PATH / 'risk_score_distribution.png'}")
print(f"  Plot 2     : {OUTPUT_PATH / 'risk_label_counts.png'}")
print(f"  Plot 3     : {OUTPUT_PATH / 'workload_timeseries.png'}")
print(f"\n  Usage      : explain_pitcher_risk('<pitcher_id>', summary)")
print('='*65)


                     FINAL PITCHER RISK SUMMARY                  
  Major (M)            → 🔴 High:   3  🟡 Moderate:  76  🟢 Low:  63
  Advanced (A)         → 🔴 High:   4  🟡 Moderate:  82  🟢 Low:  63
  Rookie (R)           → 🔴 High:   5  🟡 Moderate:  33  🟢 Low:  35

  Output CSV : /Users/pranayratan/Desktop/AdvancedLevel_SSAC Sport Med Hackathon 2026 copy/outputs/pitcher_risk_summary.csv
  Plot 1     : /Users/pranayratan/Desktop/AdvancedLevel_SSAC Sport Med Hackathon 2026 copy/outputs/risk_score_distribution.png
  Plot 2     : /Users/pranayratan/Desktop/AdvancedLevel_SSAC Sport Med Hackathon 2026 copy/outputs/risk_label_counts.png
  Plot 3     : /Users/pranayratan/Desktop/AdvancedLevel_SSAC Sport Med Hackathon 2026 copy/outputs/workload_timeseries.png

  Usage      : explain_pitcher_risk('<pitcher_id>', summary)


### 3. Biomechanical Fatigue and In-Game Stress (Advanced Analysis)
A purely volumetric approach inherently misses the physical toll exacted per-pitch. To push our injury-risk metric further, we extracted raw biomechanics data directly from the Trackman CSVs at a pitch-by-pitch level. 

Specifically, we added:
1. **High Stress Innings**: The number of innings a pitcher was forced to throw 25+ pitches (a well documented spike in acute injury tissue stress).
2. **Velocity Drop (`mean_velo_drop`)**: The difference in MPH between the first 10 pitches and the last 10 pitches of an outing.

**The Insight**: By aggregating `mean_velo_drop` over the season for our flagger pitchers, we gain a quantifiable proxy for **in-game, late-inning mechanical compensation**. When pitchers fatigue and lose velocity, kinematic efficiency degrades, heavily transferring force load directly onto the UCL and shoulder capsule. Pitchers with high `acwr_spikes` AND high `mean_velo_drop` represent the most absolute critical risk tier before entering the playoffs.